# 8.03 从零实现 RECAP：用价值函数"挑着学"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/08_vla_rl/8.03_recap.ipynb)


## 为什么要学

RECAP（RL with Experience and Corrections via Advantage-conditioned Policies）是 π\*0.6（Physical Intelligence 2025）用来让 VLA **从自己的经验中持续进步**的方法。π\*0.6 用它把做咖啡、叠衣服、组装纸箱等任务的吞吐量提升了一倍以上。

它的思路非常朴素：

1. 收集数据：示教 + 机器人自主运行（含失败）+ 人工纠正；
2. 训练一个**价值函数**，判断每一步"做得比平均好还是差"（优势）；
3. 训练 VLA 时，把"这一步优势是正/负"作为**额外的条件**输入；
4. 部署时，告诉 VLA："请按优势为正的方式做"。

和 RLT 不同，RECAP **会修改 VLA 本身**，但训练过程仍然是**监督学习**（流匹配 SFT），稳定、可扩展，不需要 on-policy 的大量交互。

本课的任务：示教质量参差不齐。一半示教是好的，另一半是**先走向错误的方块、再改正**（犯错 + 恢复）。我们看看 RECAP 能不能在**不知道哪些示教是好的**情况下，学出更好的策略。

---

## 1. 数据：一半示教会"走错再改"

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import copy, math, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.envs import ToyTableEnv, CharTokenizer, collect_table_demos
from vla_ladder.models.mini_pi0 import MiniPi0, MiniPi0Config
from vla_ladder.pipelines import get_pretrained_vlm_state, prefix_text_feature, obs_to_tensors
from vla_ladder.utils import set_seed, get_device

device = get_device()
tok = CharTokenizer()
H, T_MAX = 10, 30           # 动作块长度；评估时的时间上限（走错了就容易超时）

def collect_mixed(n, detour_frac=0.5, seed=0):
    env = ToyTableEnv(num_envs=200, seed=seed)
    reset = env.reset
    def reset_with_detour():
        obs = reset()
        env.detour = env.rng.random(env.num_envs) < detour_frac      # 这些回合中，专家先走向一个错误的方块
        return obs
    env.reset = reset_with_detour
    return collect_table_demos(env, n, noise=0.1, seed=seed)

raw = collect_mixed(2000)
ep = raw["episode"]
lens = np.bincount(ep)
detour_ep = np.array([raw["detour"][np.nonzero(ep == e)[0][0]] for e in range(len(lens))])
print(f"{len(lens)} 条示教，好示教平均 {lens[~detour_ep].mean():.1f} 步，'走错再改'的示教平均 {lens[detour_ep].mean():.1f} 步")

In [ ]:
# 构造训练张量（与 7.03 相同），并计算每一帧的"剩余步数"
F_ = len(ep)
last = np.zeros(F_, dtype=int)
for i in range(F_ - 1, -1, -1):
    last[i] = i if (i == F_ - 1 or ep[i + 1] != ep[i]) else last[i + 1]
idx_raw = np.arange(F_)[:, None] + np.arange(H)[None]
ids, msk = tok.encode(raw["instruction"])
D = {
    "image": torch.tensor(raw["image"]).to(device), "lang": torch.tensor(ids).to(device), "lang_mask": torch.tensor(msk).to(device),
    "state": torch.tensor(raw["state"]).to(device), "actions": torch.tensor(raw["action"])[torch.tensor(np.minimum(idx_raw, last[:, None]))].to(device),
    "pad": torch.tensor(idx_raw > last[:, None]).to(device),
}
steps_to_go = last - np.arange(F_) + 1

---

## 2. 价值函数：离成功还有多远？

π\*0.6 的奖励设计：**每一步 −1，直到成功**（失败则给一个很大的负值），再归一化到 $[-1, 0]$。于是价值函数 $V(o)$ 的含义就是"**从这里出发，大约还要多少步才能成功**"（取负号）。用整条轨迹的回报做回归即可（2.05 课的蒙特卡洛估计）。

$$
G_t=-\frac{\text{剩余步数}}{T_{norm}},\qquad \min_\phi\ \mathbb E\big[(V_\phi(o_t)-G_t)^2\big]
$$

我们的价值网络：**冻结的预训练 VLM** 提取前缀特征（知道目标在哪），加上本体状态，过一个 MLP。π\*0.6 的价值函数也是一个基于 VLM 的模型。

### 练习 1：价值网络

In [ ]:
vlm_state = get_pretrained_vlm_state(device=device)       # 只做过阶段 1（VLM 预训练）的权重，第一次运行约 30 秒
vlm = MiniPi0(MiniPi0Config(vocab_size=tok.vocab_size, action_horizon=H)).to(device)
vlm.load_state_dict(vlm_state); vlm.eval()
for p in vlm.parameters():
    p.requires_grad_(False)

class ValueNet(nn.Module):
    def __init__(self, vlm, hidden=256):
        super().__init__()
        self.vlm = vlm
        self.head = nn.Sequential(nn.Linear(vlm.cfg.vlm_width + 2, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def forward(self, images, lang, lang_mask, state):
        # TODO: with torch.no_grad(): feat = prefix_text_feature(self.vlm, images, lang, lang_mask)
        #       return self.head(cat([feat, state], -1)).squeeze(-1)
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
vlm_state = get_pretrained_vlm_state(device=device)
vlm = MiniPi0(MiniPi0Config(vocab_size=tok.vocab_size, action_horizon=H)).to(device)
vlm.load_state_dict(vlm_state); vlm.eval()
for p in vlm.parameters():
    p.requires_grad_(False)

class ValueNet(nn.Module):
    def __init__(self, vlm, hidden=256):
        super().__init__()
        self.vlm = vlm
        self.head = nn.Sequential(nn.Linear(vlm.cfg.vlm_width + 2, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def forward(self, images, lang, lang_mask, state):
        with torch.no_grad():
            feat = prefix_text_feature(self.vlm, images, lang, lang_mask)
        return self.head(torch.cat([feat, state], -1)).squeeze(-1)

In [ ]:
G = torch.tensor(-steps_to_go / 50.0, dtype=torch.float32, device=device)     # 归一化的回报
set_seed(0)
value = ValueNet(vlm).to(device)
opt = torch.optim.AdamW(value.head.parameters(), lr=1e-3)
for step in range(3000):
    i = torch.randint(0, F_, (256,), device=device)
    loss = ((value(D["image"][i].float() / 255, D["lang"][i], D["lang_mask"][i], D["state"][i]) - G[i]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print(f"价值函数回归误差 {loss.item():.4f}（回报的方差 {G.var().item():.4f}）")
with torch.no_grad():
    V = torch.cat([value(D["image"][j].float() / 255, D["lang"][j], D["lang_mask"][j], D["state"][j]) for j in torch.arange(F_, device=device).split(2048)])
assert loss.item() < 0.5 * G.var().item()
print("通过 ✅")

## 3. 优势与"优势指示"

用 $N$ 步回报估计每一帧的优势（π\*0.6 中也是类似的 n 步估计）：

$$
A_t=\sum_{k=0}^{N-1}r_{t+k}+V(o_{t+N})-V(o_t),\qquad r=-\tfrac{1}{50}
$$

再把优势**二值化**：$I_t=\mathbb 1[A_t>\varepsilon]$。π\*0.6 为每个任务选择阈值 $\varepsilon$，使大约 30% 的数据被标为"正"。

**直觉**："走错再改"的示教里，走向错误方块的那些帧离目标越来越远，$V$ 在下降，优势为负；好示教里每一步都在接近目标，优势为正。

### 练习 2：计算 N 步优势

In [ ]:
def n_step_advantage(V, last, N=5, r=-1 / 50.0):
    """V: (F,) 每帧的价值；last: (F,) 每帧所在回合最后一帧的下标。返回 A (F,)
    若 t+N 超过了回合末尾：剩下的步数都给奖励 r，并且末尾之后的价值视为 0（已经成功）"""
    F_ = len(V)
    t = np.arange(F_)
    # TODO:
    # k = minimum(N, last - t + 1)            实际能走的步数（包括最后一步）
    # nxt = t + k；超出回合的部分（nxt > last）价值为 0，否则为 V[nxt]
    # A = r * k + V_next - V
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def n_step_advantage(V, last, N=5, r=-1 / 50.0):
    """V: (F,) 每帧的价值；last: (F,) 每帧所在回合最后一帧的下标。返回 A (F,)
    若 t+N 超过了回合末尾：剩下的步数都给奖励 r，并且末尾之后的价值视为 0（已经成功）"""
    F_ = len(V)
    t = np.arange(F_)
    k = np.minimum(N, last - t + 1)
    nxt = t + k
    beyond = torch.tensor(nxt > last, device=V.device)
    V_next = torch.where(beyond, torch.zeros_like(V), V[torch.tensor(np.minimum(nxt, F_ - 1), device=V.device)])
    return r * torch.tensor(k, device=V.device, dtype=V.dtype) + V_next - V

In [ ]:
# ✅ 自测
A = n_step_advantage(V, last)
eps = torch.quantile(A, 0.7)
I = (A > eps).long()                                         # 约 30% 为正
det = torch.tensor(raw["detour"], device=device)
print(f"阈值 ε={eps:.4f}；正样本比例 {I.float().mean():.2f}")
print(f"好示教中被标为正的比例 {I[~det].float().mean():.2f}；'走错再改'示教中 {I[det].float().mean():.2f}")
assert I[~det].float().mean() > I[det].float().mean()
print("通过 ✅ 价值函数自动发现了哪些数据'更好'（它从未被告知哪些示教是走错的）")

plt.figure(figsize=(5, 3))
plt.hist(A[~det].cpu(), bins=60, alpha=0.6, density=True, label="good demos"); plt.hist(A[det].cpu(), bins=60, alpha=0.6, density=True, label="detour demos")
plt.axvline(eps.item(), c="k", ls="--", label="threshold"); plt.legend(); plt.xlabel("advantage"); plt.show()

---

## 4. 优势条件化的 VLA

在 π\*0.6 中，优势指示以**文字**的形式加进提示词（"Advantage: positive" / "negative"）。我们在 mini-π0 的前缀末尾加一个**可学习的条件 token**：0 = 负，1 = 正，2 = "空"（不给条件）。

训练时以 30% 的概率把条件替换成"空"，这样同一个模型也学会了**无条件**的策略，推理时可以用**无分类器引导**（CFG）：

$$
\mathbf v=\mathbf v_{\varnothing}+\beta\,(\mathbf v_{+}-\mathbf v_{\varnothing})
$$

$\beta=1$ 即直接以"正"为条件；$\beta>1$ 更强调"正"的行为。

### 练习 3：在前缀中加入条件 token

In [ ]:
class AdvMiniPi0(MiniPi0):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.adv_embed = nn.Embedding(3, cfg.vlm_width)      # 0 = 负，1 = 正，2 = 空
        self.cond = None                                      # (B,) 长整型，调用前设置

    def embed_prefix(self, images, lang_tokens, lang_masks, state=None):
        embs, pad, att = super().embed_prefix(images, lang_tokens, lang_masks, state)
        # TODO: c = adv_embed(self.cond)[:, None] * sqrt(vlm_width)
        #       在序列末尾拼上 c；pad 拼上 True；att 拼上 0（与前缀同一块，双向）
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class AdvMiniPi0(MiniPi0):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.adv_embed = nn.Embedding(3, cfg.vlm_width)
        self.cond = None

    def embed_prefix(self, images, lang_tokens, lang_masks, state=None):
        embs, pad, att = super().embed_prefix(images, lang_tokens, lang_masks, state)
        c = self.adv_embed(self.cond)[:, None] * math.sqrt(self.cfg.vlm_width)
        return torch.cat([embs, c], 1), torch.cat([pad, torch.ones_like(pad[:, :1])], 1), torch.cat([att, torch.zeros_like(att[:, :1])], 1)

In [ ]:
def train_policy(model, cond_fn=None, idx_pool=None, steps=8000, drop=0.3):
    """cond_fn(i) -> 条件 (B,)；idx_pool：只从这些帧中采样（用于"只用好示教"的对照组）"""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps, pct_start=0.05)
    model.train(); t0 = time.time()
    for step in range(steps):
        i = torch.randint(0, F_, (128,), device=device) if idx_pool is None else idx_pool[torch.randint(0, len(idx_pool), (128,), device=device)]
        if cond_fn is not None:
            c = cond_fn(i)
            model.cond = torch.where(torch.rand(len(c), device=device) < drop, torch.full_like(c, 2), c)
        per = model(D["image"][i].float() / 255, D["lang"][i], D["lang_mask"][i], D["state"][i], D["actions"][i])
        v = (~D["pad"][i]).float()[..., None]
        loss = (per * v).sum() / (v.sum() * per.shape[-1])
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if step % 2000 == 0 or step == steps - 1:
            print(f"    step {step:5d}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)")
    model.eval()
    return model

@torch.no_grad()
def evaluate(model, cond=None, beta=None, n=500, seed=123):
    """返回 (成功次数, 回合数, 成功回合的平均步数)。beta 不为 None 时使用 CFG"""
    e = ToyTableEnv(num_envs=n, seed=seed, max_steps=T_MAX)
    obs = e.reset(); steps = np.full(n, T_MAX)
    while not e.done.all():
        I_, L_, M_, S_ = obs_to_tensors(obs, tok, device)
        if beta is None:
            if cond is not None:
                model.cond = torch.full((n,), cond, device=device)
            chunk = model.sample_actions(I_, L_, M_, S_)
        else:                                              # CFG：分别以"正"和"空"为条件，组合速度
            noise = torch.randn(n, H, 2, device=device)
            model.cond = torch.full((n,), 1, device=device); cp, pp, _ = model.prefix_cache(I_, L_, M_, S_)
            model.cond = torch.full((n,), 2, device=device); cn, pn, _ = model.prefix_cache(I_, L_, M_, S_)
            x, t, dt = noise, 1.0, -0.1
            while t >= -dt / 2:
                tt = torch.full((n,), t, device=device)
                v_pos, v_null = model.denoise_step(S_, pp, cp, x, tt), model.denoise_step(S_, pn, cn, x, tt)
                x = x + dt * (v_null + beta * (v_pos - v_null)); t += dt
            chunk = x
        chunk = chunk.clamp(-1, 1).cpu().numpy()
        for k in range(5):
            prev = e.done.copy(); obs, _, _, info = e.step(chunk[:, k]); steps[e.success & ~prev] = e.t
    return int(info["success"].sum()), n, float(steps[e.success].mean())

### 训练三个模型并比较（GPU 约 5~6 分钟）

In [ ]:
cfg = MiniPi0Config(vocab_size=tok.vocab_size, action_horizon=H)
results = {}

print("① 普通 BC（混合数据，好坏不分）")
set_seed(0); bc = MiniPi0(cfg).to(device); bc.load_state_dict(vlm_state)
results["BC (mixed)"] = evaluate(train_policy(bc))

print("② RECAP（优势条件化，混合数据）")
set_seed(0); recap = AdvMiniPi0(cfg).to(device); recap.load_state_dict(vlm_state, strict=False)
train_policy(recap, cond_fn=lambda i: I[i])
results["RECAP (+)"] = evaluate(recap, cond=1)
results["RECAP (-)"] = evaluate(recap, cond=0)
results["RECAP CFG β=1.5"] = evaluate(recap, beta=1.5)

print("③ 对照：只用好示教的 BC（需要事先知道哪些示教是好的，现实中往往做不到）")
set_seed(0); oracle = MiniPi0(cfg).to(device); oracle.load_state_dict(vlm_state)
results["BC (good demos only)"] = evaluate(train_policy(oracle, idx_pool=torch.nonzero(~det)[:, 0]))

In [ ]:
from math import sqrt

def wilson_ci(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

print(f"{'方法':<24}{'成功率':>8}{'95% 置信区间':>18}{'成功回合平均步数':>16}")
for name, (k, n, st) in results.items():
    lo, hi = wilson_ci(k, n)
    print(f"{name:<24}{k / n:>8.3f}{f'[{lo:.3f}, {hi:.3f}]':>18}{st:>16.1f}")
assert results["RECAP (+)"][0] > results["RECAP (-)"][0]

### 读懂结果

（数字会随随机种子波动几个百分点，请结合置信区间来看。）

- **RECAP(+) 优于普通 BC**：模型学会了区分"好的做法"和"走错的做法"，部署时只按好的方式做。它**不需要知道哪些示教是好的**，这个信息完全来自价值函数。
- **RECAP(−) 最差**：以"负"为条件时，模型会去模仿走错的行为。这说明条件 token 确实控制了行为，是一个很好的**正确性检查**。
- **RECAP(+) 接近"只用好示教"的上限**：在真实场景中，你通常没法提前把数据分成好坏两堆（尤其是自主运行产生的大量数据），RECAP 让价值函数自动完成这件事，而且坏数据也没有浪费：它让模型学会了"什么是坏的"。
- **CFG**：在我们的玩具任务上 β>1 不一定带来额外提升；π\*0.6 报告在真实任务上适当的 β 有帮助。

> **完整的 RECAP 是一个迭代过程**：部署 → 收集自主运行数据（成功和失败）+ 人工纠正 → 更新价值函数 → 重新计算优势、重新训练 VLA → 再部署。每一轮的数据都来自更好的策略。本课只做了一轮。

---

## 5. RLT 与 RECAP 怎么选？

| | RLT（8.02） | RECAP（8.03） |
|---|---|---|
| 修改 VLA 吗 | 否（冻结） | 是（重新做 SFT） |
| 学习方式 | 在线 off-policy actor-critic | 离线、迭代的监督学习 |
| 擅长 | 精细的"最后一厘米"（插接、对准） | 整体效率、从失败中恢复、长任务 |
| 数据 | 真机交互 + 人工接管，几百到几千个回合 | 大量自主运行数据 + 人工纠正 |
| 风险 | actor 偏离太远（靠 BC 正则和小 σ 控制） | 价值函数不准会给错优势 |
| 算力 | 小（训练小网络） | 大（每轮都要训练 VLA） |

两者可以结合：先用 RECAP 把整体流程做好，再用 RLT 精修关键的插入阶段。

## 小结

- RECAP = 价值函数 + 优势二值化 + 优势条件化的 SFT（+ 可选 CFG）。
- 价值函数能自动识别数据的好坏，模型学会"按好的方式做"。
- 用"以负为条件"做正确性检查；用置信区间判断改进是否可信（8.04 课）。